# ⭐ **Informed Search & $A*$ Search — Theory Overview**

## 1. Introduction to Informed Search

Informed search algorithms use **additional knowledge** (heuristics) about the problem domain to guide the search process more efficiently than uninformed methods such as BFS or DFS.

The core idea is simple:

> **Use domain knowledge to prioritize nodes that are more likely to lead to the goal.**

This reduces the number of explored states and often yields optimal solutions with significantly less computational effort.

$A*$ is the most widely used informed search algorithm because it combines **optimality**, **completeness**, and **efficiency**.

---

## 2. The Cost Structure in Informed Search

In informed search, each node has an associated **evaluation function** that determines how desirable it is to expand that node next.

$A*$ uses the following evaluation function:

$$
f(n) = g(n) + h(n)
$$

Where:

- **$g(n)$** = the *actual cost* from the initial state to node *n*  
- **$h(n)$** = the *estimated cost* from node *n* to the goal  
- **$f(n)$** = the *estimated total cost* of the cheapest solution passing through *n*

This combination allows $A*$ to balance:

- **exploration of known good paths** (via `$g(n)$`)
- **exploration of promising future paths** (via `$h(n)$`)

---

## 3. Understanding `$g(n)$` — The Path Cost So Far

`$g(n)$` represents the **accumulated cost** from the start node to the current node.

Examples of `$g(n)$` depending on the domain:

- number of steps taken  
- distance traveled (e.g., meters in your Metro CDMX graph)  
- time elapsed  
- energy consumed  
- any additive cost defined by the problem  

$A*$ requires that **edge costs are non-negative**, ensuring that `$g(n)$` always increases as the search progresses.

---

## 4. Understanding `$h(n)$` — The Heuristic Estimate

`$h(n)$` is a **heuristic function** that estimates the cost from node *$n$* to the goal.

$A$ heuristic must satisfy:

### **Admissibility**

$$
h(n) \leq h^*(n)
$$

It must **never overestimate** the true cost to the goal.

### **Consistency (Monotonicity)**

$$
h(n) \leq c(n, n') + h(n')
$$

Where `$c(n, n')$` is the cost of moving from `$n$` to `$n'$`.

Consistency guarantees:

- optimality  
- no need to revisit nodes  
- `$f(n)$` values are non-decreasing along any path  

---

## 5. The $A*$ Evaluation Function

$A*$ selects the next node to expand based on:

$$
f(n) = g(n) + h(n)
$$

Interpretation:

- **$g(n)$**: “What I have already paid.”
- **$h(n)$**: “What I expect to pay to finish.”
- **$f(n)$**: “Total estimated cost of this route.”

$A*$ always expands the node with the **lowest $f(n)$** in the frontier.

This makes $A*$ both:

- **goal-directed** (thanks to `$h(n)$`)  
- **cost-aware** (thanks to `$g(n)$`)  

---

## 6. Why $A*$ Is Optimal

$A*$ is guaranteed to find an optimal solution if:

1. The heuristic is **admissible**  
2. All edge costs are **non-negative**

If the heuristic is also **consistent**, $A*$ becomes even more efficient because:

- it never reopens nodes  
- the `$f(n)$` values along any path are monotonic  
- the search behaves like Dijkstra’s algorithm with a “hint” toward the goal  

---

## 7. $A*$ in Weighted Graphs (like Metro CDMX)

In this case:

- **$g(n)$** = sum of real walking distances (meters) between stations  
- **$h(n)$** = straight-line (Euclidean) distance to the goal station  
- **$f(n)$** = estimated total walking distance of the route

This makes $A*$ extremely effective for:

- shortest paths  
- navigation  
- transportation networks  
- any domain where distances matter  

---

## 8. Summary

$A*$ works by combining:

- **the cost so far** (`$g(n)$`)  
- **the estimated cost to finish** (`$h(n)$`)  

into a single value:

$$
f(n) = g(n) + h(n)
$$

This allows $A*$ to explore paths that are both:

- **cheap so far**, and  
- **likely to lead to a cheap solution overall**

When the heuristic is admissible and consistent, $A*$ is:

- **complete**  
- **optimal**  
- **efficient**  

In [1]:
import sys
import os
import json
from dotenv import load_dotenv
import os

# Obtener la ruta absoluta del proyecto (dos niveles arriba)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Agregar al path
sys.path.append(PROJECT_ROOT)
ENV_PATH = os.path.join(PROJECT_ROOT, ".env")

load_dotenv(ENV_PATH)

API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")

print("Project root added:", PROJECT_ROOT)

Project root added: C:\Users\odver\OneDrive\Documentos\MAC-Ayrton94\2026-2\git\MAC-Ayrton94\Sistemas-Inteligentes


In [2]:
## Imports del proyecto
from src.catalog_loader import load_metro_graph
from src.CDMX_metro_weighted_graph import build_weighted_graph, get_station_coords, straight_line_distance
from src.helper_funcs import preview

In [3]:
metro_graph = load_metro_graph(folder="../metro")
print("Preview, first 5 keys")
preview(metro_graph)

163 stations were uploaded from '../metro' directory.
Preview, first 5 keys


{'Observatorio': "{'Tacubaya': 1}",
 'Tacubaya': "{'Observatorio': 1, 'Juanacatlán': 1, 'Constituyentes': 1, 'San Pedro de los Pin...",
 'Juanacatlán': "{'Tacubaya': 1, 'Chapultepec': 1}",
 'Chapultepec': "{'Juanacatlán': 1, 'Sevilla': 1}",
 'Sevilla': "{'Chapultepec': 1, 'Insurgentes': 1}"}

In [4]:
# Run the following code to generate the weighted graph
# weighted_graph = build_weighted_graph(API_KEY, metro_graph)

# Otherwise if you already have it stored physically in JSON format
with open("../output/CDMX_metro_weighted.json", "r", encoding="utf-8") as f:
    weighted_graph = json.load(f)

print("Preview, first 5 keys")
preview(weighted_graph)

Preview, first 5 keys


{'Observatorio': "{'Tacubaya': 2942}",
 'Tacubaya': "{'Observatorio': 2942, 'Juanacatlán': 1579, 'Constituyentes': 1427, 'San Pedro d...",
 'Juanacatlán': "{'Tacubaya': 1579, 'Chapultepec': 1939}",
 'Chapultepec': "{'Juanacatlán': 1939, 'Sevilla': 1087}",
 'Sevilla': "{'Chapultepec': 1087, 'Insurgentes': 947}"}

In [5]:
# Build the directory to CDMX_metro_weighted.json
# OUTPUT_PATH = os.path.join(PROJECT_ROOT, "output", "CDMX_metro_weighted.json")

# Store the weighted_graph
# with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
#     json.dump(weighted_graph, f, indent=4, ensure_ascii=False)

# print("Weighted graph saved to:", OUTPUT_PATH)

In [6]:
# To generate the coordinates for the first time
# metro_coords = get_station_coords(weighted_graph, API_KEY)

# with open("../output/CDMX_metro_coords.json", "w", encoding="utf-8") as f:
#     json.dump(metro_coords, f, indent=4, ensure_ascii=False)

# To read if it was already generated
with open("../output/CDMX_metro_coords.json", "r", encoding="utf-8") as f:
    metro_coords = json.load(f)

print("Preview, first 5 keys")
preview(metro_coords)

Preview, first 5 keys


{'Observatorio': '[19.39828, -99.2004]',
 'Tacubaya': '[19.401932, -99.187355]',
 'Juanacatlán': '[19.4128272, -99.18181229999999]',
 'Chapultepec': '[19.420899, -99.1755634]',
 'Sevilla': '[19.421501, -99.1710272]'}

In [11]:
# To calculate a straight line to a specified target station
target_station="Canal de San Juan"

try:
    with open(f"../output/CDMX_{target_station}_straight_line_haversine.json", "r", encoding="utf-8") as f:
        straight_line = json.load(f)
except:
    straight_line = straight_line_distance(target_station=target_station, coords=metro_coords, method="haversine")
    with open(f"../output/CDMX_{target_station}_straight_line_haversine.json", "w", encoding="utf-8") as f:
        json.dump(straight_line, f, indent=4, ensure_ascii=False)

print("Printing preview: ", preview(straight_line))    

Printing preview:  {'Observatorio': '15676.263045861686', 'Tacubaya': '14225.832866782943', 'Juanacatlán': '13611.661176653839', 'Chapultepec': '12920.446588865922', 'Sevilla': '12416.61802440624'}


In [13]:
class Problem:
    def __init__(self, initial, goal):
        self.initial = initial
        self.goal = goal

    def actions(self, state):
        raise NotImplementedError

    def result(self, state, action):
        raise NotImplementedError

    def is_goal(self, state):
        return self.goal == state

    def action_cost(self, state1, action, state2):
        """
        Returns the actual cost of moving from state1 to state2.
        A* will accumulate these costs as g(n).
        """
        return 1  # override in subclasses

    def h(self, state):
        """
        Heuristic estimate of the cost from 'state' to the goal.
        A* uses this as h(n).
        Must be admissible (never overestimate).
        """
        return 0  # override in subclasses


## GraphASProblem

In [14]:
class GraphASProblem(Problem):
    def __init__(self, initial, goal, graph, heuristic_table):
        super().__init__(initial, goal)
        self.graph = graph
        self.heuristic_table = heuristic_table  # diccionario de heurística

    def actions(self, state):
        return list(self.graph[state].keys())

    def result(self, state, action):
        return action

    def action_cost(self, state1, action, state2):
        # ONLY the real cost of the graph
        return self.graph[state1][state2]

    def h(self, state):
        # ONLY heuristic
        return self.heuristic_table[state]

## Node Class Overview

The `Node` class represents a single node in a search tree.  
It encapsulates all the information needed by search algorithms to track states, reconstruct solution paths, and compute cumulative costs.  
Each node corresponds to a specific state in the problem domain and maintains links to its parent, the action that generated it, and the total path cost from the initial state.

### Key Responsibilities

- **Store the current state** of the search.
- **Maintain a reference to the parent node**, enabling reconstruction of the full solution path.
- **Record the action** that led from the parent state to the current state.
- **Accumulate the path cost**, which is essential for cost‑based search algorithms such as Uniform Cost Search and A*.
- **Generate child nodes** based on the problem’s transition model.

### Class Structure

- `__init__(state, parent=None, action=None, path_cost=0)`  
  Initializes a node with:
  - `state`: the current state in the search space  
  - `parent`: the node from which this one was generated  
  - `action`: the action applied to reach this state  
  - `path_cost`: the cumulative cost from the initial node  

- `path()`  
  Reconstructs the sequence of states from the initial node to the current node.  
  It follows parent links backward and returns the path in forward order.

- `expand(problem)`  
  Generates all successor nodes reachable from the current state.  
  It retrieves the available actions from the problem and creates a child node for each one.

- `child_node(problem, action)`  
  Creates a new node representing the result of applying the given action.  
  It computes:
  - the next state via `problem.result`  
  - the step cost via `problem.action_cost`  
  - the updated cumulative path cost  

### Usage

The `Node` class is fundamental for search algorithms such as BFS, DFS, Uniform Cost Search, A*, and others.  
It provides the structure needed to explore the search space, track progress, and reconstruct optimal or feasible solutions.


In [15]:
class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0, h=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost  # g(n)
        self.h = h                  # h(n)
        self.f = path_cost + h      # f(n)

    def path(self):
        lista_path = []
        node = self
        while node:
            lista_path.append(node.state)
            node = node.parent
        return lista_path[::-1]

    def expand(self, problem):
        lista = []
        for action in problem.actions(self.state):
            lista.append(self.child_node(problem, action))
        return lista

    def child_node(self, problem, action):
        next_state = problem.result(self.state, action)
        step_cost = problem.action_cost(self.state, action, next_state)
        g = self.path_cost + step_cost
        h = problem.h(next_state)
        return Node(next_state, self, action, g, h)

    def __lt__(self, other):
        # Necesario para heapq (priority queue)
        return self.f < other.f

## a_star_search Function Overview


In [16]:
import heapq

def a_star_search(problem):
    # Nodo inicial
    start = Node(
        state=problem.initial,
        parent=None,
        action=None,
        path_cost=0,
        h=problem.h(problem.initial)
    )

    # Frontier como priority queue ordenada por f(n)
    frontier = []
    heapq.heappush(frontier, start)

    # Estados ya explorados con su mejor costo g(n)
    explored = {}

    while frontier:
        node = heapq.heappop(frontier)

        # Si llegamos a la meta, regresamos el nodo
        if problem.is_goal(node.state):
            return node

        # Si ya vimos este estado con menor costo, lo ignoramos
        if node.state in explored and explored[node.state] <= node.path_cost:
            continue

        explored[node.state] = node.path_cost

        # Expandir hijos
        for child in node.expand(problem):
            # Si el hijo ya fue explorado con menor costo, ignorar
            if child.state in explored and explored[child.state] <= child.path_cost:
                continue

            heapq.heappush(frontier, child)

    return None

## $A*$ Search Example: Romania

In [19]:
problem = GraphASProblem(initial="Vallejo", goal="Canal de San Juan", graph=weighted_graph, heuristic_table=straight_line)
node = a_star_search(problem)
node.path()

['Vallejo',
 'Instituto del Petróleo',
 'Autobuses del Norte',
 'La Raza',
 'Misterios',
 'Valle Gómez',
 'Consulado',
 'Canal del Norte',
 'Morelos',
 'San Lázaro',
 'Moctezuma',
 'Balbuena',
 'Boulevard Puerto Aéreo',
 'Gómez Farías',
 'Zaragoza',
 'Pantitlán',
 'Agrícola Oriental',
 'Canal de San Juan']